# MLP and Transformer Split-Horizon Forecasting

This notebook uses one shared BasicTS forecasting data pipeline for both models. The MLP forecasts steps 1-6, the Transformer forecasts steps 7-12, and their 6-step outputs are concatenated into one 12-step hybrid forecast.


## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [ ]:
import os 
import sys
from pathlib import Path
#root contains the path to the basicts
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
#move python working folder
os.chdir(ROOT)
# the path to src is src_path
src_path = ROOT / "src"
#if src_path is not in the system path, add it to the system path
#system path is added to python search. 
# This allows us to import modules from the src 
# folder without having to specify the full path.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))



## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, train/val/test split, and batch format. The shared dataset keeps the full 12-step target window, while each split-horizon model trains on its own 6-step slice.


In [ ]:
import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode
DATASET_NAME = "ETTh1"
#most papers use 96 input length
INPUT_LEN = 96
#most papers use 96, 192, 336, and 720 input length
FULL_OUTPUT_LEN = 12

SPLIT_OUTPUT_LEN = 6
#etthl has 7 variables
NUM_FEATURES = 7
#Batch sizes like 16, 32, and 64 are normal. 32 is a safe default.
BATCH_SIZE = 32
# common for testing
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
# Use a fresh checkpoint namespace so BasicTS does not auto-resume from old/corrupt files.
RUN_TAG = "mixed_v2"
# this is the shared settings dictionary that both models use
SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": True,
}

# The full 12-step standalone models only need BasicTS test_metrics.json.
# Keeping save_results=False avoids Windows memmap file-lock errors during evaluation.
FULL_12_STEP_CONFIG = dict(SHARED_CONFIG)
FULL_12_STEP_CONFIG["save_results"] = False


## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [ ]:
# for the split forecasitting model, we need to create a custom taskflow that slices the targets and target masks to the desired output length      
#start with the forecasting taskflwo and mofify it 
# BasicTSForecastingTaskFlow is the default data-prep worker.
# It prepares each forecasting batch before the model uses it.
# start with the deault taskflow and then add a change 
class SplitHorizonForecastingTaskFlow(BasicTSForecastingTaskFlow):
    #adding a setting called targest slide whchi is a variable that is used to slice the targets
    # tells the taskflow which targets to keep
    # need self because we need acresss to this specific object
    #creates a variables incide the class object 
    def __init__(self, target_slice):
        self.target_slice = target_slice
    # preprocess 
    def preprocess(self, runner, data):
        #normal work
        data = super().preprocess(runner, data)
        #cuts target values
        data["targets"] = data["targets"][:, self.target_slice, :]
        # tells basicts which target values are valid
        data["targets_mask"] = data["targets_mask"][:, self.target_slice, :]
        return data


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }

# checks to see if the inputs are the right shape the targest are sliced and the prediction matches targer
def preview_shapes(cfg, model_name):
    #build the dataset using basicts
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    # puts the dataset into batches gets ready for batches
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    #This grabs the first batch.
    raw_batch = _float_batch(next(iter(train_loader)))
    #creates the scaler and fits it to the training data  
    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)
    # fake runner so the pasicts 
    class PreviewRunner:
        pass
    # Create a fake runner that has cfg and scaler, because taskflow.preprocess expects a runner object.
    runner = PreviewRunner() 
    runner.cfg = cfg
    runner.scaler = scaler
    #prepares the batch normally
    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))\
    #build the model from the config and switch it to evaluation mode
    model = cfg.model(cfg.model_config)
    model.eval()
    # o not track gradient as they are only needed for trainig 
    with torch.no_grad():
        #sends processed inputs into the model
        prediction = model(processed_batch["inputs"])
        # then the model outputs future values
        # did the model return a dictionary
        if isinstance(prediction, dict):
            # if it did this extracts onlt the prediction tensor
            prediction = prediction["prediction"]
    #print the shapres to check that the data and model match before the training
    print(f"{model_name} raw inputs shape:       ", tuple(raw_batch["inputs"].shape))
    print(f"{model_name} raw target shape:       ", tuple(raw_batch["targets"].shape))
    print(f"{model_name} processed inputs shape: ", tuple(processed_batch["inputs"].shape))
    print(f"{model_name} target shape:           ", tuple(processed_batch["targets"].shape))
    print(f"{model_name} prediction shape:       ", tuple(prediction.shape))
    #checks
    assert tuple(raw_batch["targets"].shape) == (cfg.batch_size, FULL_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["targets"].shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


## 4. MLP Model

The MLP flattens `[batch_size, input_len, num_features]`, passes the flat vector through linear layers, and reshapes the head output back to `[batch_size, 6, num_features]`. In this split-horizon hybrid, the MLP is responsible for forecast steps 1-6.


In [ ]:
#simple mlp forecasting model
#nnmodule is the base class for all pytorch models 
class SimpleMLPForecaster(nn.Module):
    # innit runs when the model is creates 
    def __init__(self, config):
        #calls the parent class setup code
        super().__init__()
        #inportant settings
        self.input_len = config.input_len
        self.output_len = config.output_len
        self.num_features = config.num_features
        flat_input = self.input_len * self.num_features
        flat_output = self.output_len * self.num_features
        #flatten  it into a single vector to the mlp work
        self.net = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(flat_input, config.hidden_size),
            #adds nonlinarity
            nn.GELU(),
            #reduce overfitting by randomly setting some values to zero during training
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, flat_output),
        )
    #forward pass 
    def forward(self, inputs):
        prediction = self.net(inputs)
        return prediction.view(inputs.size(0), self.output_len, self.num_features)


## 5. MLP Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [ ]:
#tells basicts hwo to bild and train the MLP
mlp_model_config = BasicTSModelConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=256,
    dropout=0.1,
)

#full basicts training config for MLP
mlp_cfg = BasicTSForecastingConfig(
    model=SimpleMLPForecaster,
    model_config=mlp_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(0, SPLIT_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/SimpleMLPForecaster/{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    **SHARED_CONFIG,
)

# Standalone MLP trained to forecast all 12 steps.
mlp_full_model_config = BasicTSModelConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=256,
    dropout=0.1,
)

mlp_full_cfg = BasicTSForecastingConfig(
    model=SimpleMLPForecaster,
    model_config=mlp_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/SimpleMLPForecaster/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

mlp_cfg, mlp_full_cfg


## 6. MLP Shape Test

Run this before training. The model prediction and processed target lines must be `(batch_size, 6, num_features)`, while the raw target line remains `(batch_size, 12, num_features)`.


In [ ]:
#checker
mlp_batch, mlp_prediction = preview_shapes(mlp_cfg, "MLP")


## 7. Train the MLP

This is the first training run. Leave `RUN_MLP_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [ ]:
#training
RUN_MLP_TRAINING = False
RUN_MLP_12_STEP_TRAINING = False

if RUN_MLP_TRAINING:
    BasicTSLauncher.launch_training(mlp_cfg)
else:
    print("MLP split-horizon training skipped. Set RUN_MLP_TRAINING = True to train.")

if RUN_MLP_12_STEP_TRAINING:
    BasicTSLauncher.launch_training(mlp_full_cfg)
else:
    print("MLP 12-step training skipped. Set RUN_MLP_12_STEP_TRAINING = True to train.")


## 8. Transformer Model

After the MLP shape check works, use the repo's existing `iTransformerForForecasting`. It receives the same `[batch_size, input_len, num_features]` input and returns `[batch_size, 6, num_features]`. In this split-horizon hybrid, the Transformer is responsible for forecast steps 7-12.


In [ ]:
#transfoemr model build it
transformer_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(SPLIT_OUTPUT_LEN, FULL_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    **SHARED_CONFIG,
)

# Standalone Transformer trained to forecast all 12 steps.
transformer_full_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_full_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

transformer_cfg, transformer_full_cfg


## 9. Transformer Shape Test

Run this after the MLP section works. It uses the same shared data pipeline and checks that the Transformer predicts only its 6-step split-horizon target.


In [ ]:
#check the shapes
transformer_batch, transformer_prediction = preview_shapes(transformer_cfg, "Transformer")


## 10. Train the Transformer

Use the same dataset, scaler, preprocessing, and `input_len` as the MLP. The shared dataset still contains the full 12-step target window, but the Transformer taskflow slices that target to steps 7-12.


In [ ]:
#train trasnfoemr
RUN_TRANSFORMER_TRAINING = False
RUN_TRANSFORMER_12_STEP_TRAINING = False

if RUN_TRANSFORMER_TRAINING:
    BasicTSLauncher.launch_training(transformer_cfg)
else:
    print("Transformer split-horizon training skipped. Set RUN_TRANSFORMER_TRAINING = True after the MLP works.")

if RUN_TRANSFORMER_12_STEP_TRAINING:
    BasicTSLauncher.launch_training(transformer_full_cfg)
else:
    print("Transformer 12-step training skipped. Set RUN_TRANSFORMER_12_STEP_TRAINING = True to train.")


## 11. Hybrid Prediction: MLP Steps 1-6, Transformer Steps 7-12

This is a true split-horizon hybrid. The MLP predicts only the first 6 forecast steps, the Transformer predicts only the next 6 forecast steps from the same input window, and the final hybrid prediction is the time-axis concatenation of those two 6-step outputs.


In [ ]:
import numpy as np

#checks that the mlp and transformer both used the same batch
assert torch.equal(mlp_batch["inputs"], transformer_batch["inputs"])

# Split-horizon hybrid: MLP predicts steps 1-6, Transformer predicts steps 7-12.
mlp_pred = mlp_prediction
transformer_pred = transformer_prediction
#this combines the two step prediction into one 12 step hybrui prediction
hybrid_pred = torch.cat([mlp_pred, transformer_pred], dim=1)
#combines the targests together
hybrid_targets = torch.cat([mlp_batch["targets"], transformer_batch["targets"]], dim=1)
# checks 
print("MLP prediction shape:", tuple(mlp_pred.shape))
print("Transformer prediction shape:", tuple(transformer_pred.shape))
print("Hybrid prediction shape:", tuple(hybrid_pred.shape))
print("Target shape:", tuple(hybrid_targets.shape))

assert tuple(mlp_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(transformer_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_pred.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_targets.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)

#find the last saved prediction
def latest_prediction_file(cfg):
    prediction_files = sorted(
        Path(cfg.ckpt_save_dir).glob("*/test_results/prediction.npy"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction.npy found under {cfg.ckpt_save_dir}. Train/evaluate this model with save_results=True first."
        )
    return prediction_files[0]

#load the prediction arrays
def load_basicts_array(path, shape):
    # BasicTS writes these files as raw memmaps, even though the file names end in .npy.
    array = np.memmap(path, dtype=np.float32, mode="r", shape=shape)
    return np.asarray(array)

#loads the largets
def load_basicts_prediction_and_targets(cfg, output_len):
    prediction_path = latest_prediction_file(cfg)
    targets_path = prediction_path.parent / "targets.npy"
    if not targets_path.exists():
        raise FileNotFoundError(f"No targets.npy found next to {prediction_path}")

    test_dataset = Builder._build_dataset(cfg, BasicTSMode.TEST)
    shape = (len(test_dataset), output_len, NUM_FEATURES)
    prediction = load_basicts_array(prediction_path, shape)
    targets = load_basicts_array(targets_path, shape)
    return prediction, targets


mlp_test_pred, mlp_test_targets = load_basicts_prediction_and_targets(mlp_cfg, SPLIT_OUTPUT_LEN)
transformer_test_pred, transformer_test_targets = load_basicts_prediction_and_targets(transformer_cfg, SPLIT_OUTPUT_LEN)
#bombines the test predictions
hybrid_test_pred = np.concatenate([mlp_test_pred, transformer_test_pred], axis=1)
hybrid_test_targets = np.concatenate([mlp_test_targets, transformer_test_targets], axis=1)

assert mlp_test_pred.shape == mlp_test_targets.shape
assert transformer_test_pred.shape == transformer_test_targets.shape
assert mlp_test_pred.shape == transformer_test_pred.shape
assert hybrid_test_pred.shape == hybrid_test_targets.shape
assert hybrid_test_pred.shape[1] == FULL_OUTPUT_LEN

hybrid_save_path = Path("checkpoints/hybrid_split_horizon_mlp_steps_1_6_transformer_steps_7_12_ETTh1_96_12_prediction.npy")
np.save(hybrid_save_path, hybrid_test_pred)


## 12. MAE/MSE Comparison

This cell computes MAE/MSE directly from the saved predictions and targets. The MLP is compared only against target steps 1-6, the Transformer only against target steps 7-12, and the hybrid against the full 12-step target.


In [ ]:
#use math to compute metrics
def compute_metrics(prediction, targets):
    return {
        "MAE": float(np.mean(np.abs(prediction - targets))),
        "MSE": float(np.mean((prediction - targets) ** 2)),
    }

def latest_metrics_file(cfg):
    metrics_files = sorted(
        Path(cfg.ckpt_save_dir).glob("*/test_metrics.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not metrics_files:
        raise FileNotFoundError(f"No test_metrics.json found under {cfg.ckpt_save_dir}. Train this model first.")
    return metrics_files[0]


def load_test_metrics(cfg):
    metrics_path = latest_metrics_file(cfg)
    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    return metrics.get("overall", metrics)


# Compare the split models, hybrid, and standalone 12-step models.
comparison = {
    "MLP split steps 1-6": compute_metrics(mlp_test_pred, mlp_test_targets),
    "Transformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Hybrid split steps 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "MLP full steps 1-12": load_test_metrics(mlp_full_cfg),
    "Transformer full steps 1-12": load_test_metrics(transformer_full_cfg),
}

print("MLP target shape:", mlp_test_targets.shape)
print("Transformer target shape:", transformer_test_targets.shape)
print("Hybrid target shape:", hybrid_test_targets.shape)
print("MLP full 12-step metrics file:", latest_metrics_file(mlp_full_cfg))
print("Transformer full 12-step metrics file:", latest_metrics_file(transformer_full_cfg))

for model_name, metrics in comparison.items():
    print(model_name)
    for metric_name in ["MAE", "MSE"]:
        print(f"  {metric_name}: {metrics[metric_name]:.6f}")


## 13. Efficiency Comparison

This cell compares model size and average batch prediction time. The MLP timing is for its 6-step prediction, the Transformer timing is for its 6-step prediction, and the hybrid timing is the sum of running both 6-step models once.


In [ ]:

import time

#counts model size
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

#takes one batch and runs the model many times
def time_model_prediction(model, batch, expected_output_len, repeats=50, warmup=5):
    device = next(model.parameters()).device
    inputs = batch["inputs"].to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inputs)

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            prediction = model(inputs)
    end = time.perf_counter()

    if isinstance(prediction, dict):
        prediction = prediction["prediction"]
    assert tuple(prediction.shape) == (inputs.size(0), expected_output_len, NUM_FEATURES)
    return (end - start) / repeats


mlp_efficiency_model = mlp_cfg.model(mlp_cfg.model_config)
transformer_efficiency_model = transformer_cfg.model(transformer_cfg.model_config)
mlp_full_efficiency_model = mlp_full_cfg.model(mlp_full_cfg.model_config)
transformer_full_efficiency_model = transformer_full_cfg.model(transformer_full_cfg.model_config)

mlp_params = count_trainable_parameters(mlp_efficiency_model)
transformer_params = count_trainable_parameters(transformer_efficiency_model)
mlp_full_params = count_trainable_parameters(mlp_full_efficiency_model)
transformer_full_params = count_trainable_parameters(transformer_full_efficiency_model)
hybrid_params = mlp_params + transformer_params

mlp_time = time_model_prediction(mlp_efficiency_model, mlp_batch, SPLIT_OUTPUT_LEN)
transformer_time = time_model_prediction(transformer_efficiency_model, transformer_batch, SPLIT_OUTPUT_LEN)
mlp_full_time = time_model_prediction(mlp_full_efficiency_model, mlp_batch, FULL_OUTPUT_LEN)
transformer_full_time = time_model_prediction(transformer_full_efficiency_model, transformer_batch, FULL_OUTPUT_LEN)
hybrid_time = mlp_time + transformer_time

print("MLP parameters:", mlp_params)
print("Transformer parameters:", transformer_params)
print("Hybrid parameters:", hybrid_params)
print("MLP full 12-step parameters:", mlp_full_params)
print("Transformer full 12-step parameters:", transformer_full_params)

print(f"MLP 6-step avg prediction time: {mlp_time:.6f} seconds")
print(f"Transformer 6-step avg prediction time: {transformer_time:.6f} seconds")
print(f"Hybrid avg prediction time: {hybrid_time:.6f} seconds")
print(f"MLP full 12-step avg prediction time: {mlp_full_time:.6f} seconds")
print(f"Transformer full 12-step avg prediction time: {transformer_full_time:.6f} seconds")

rows = [
    ("MLP split 1-6", comparison["MLP split steps 1-6"], mlp_params, mlp_time),
    ("Transformer split 7-12", comparison["Transformer split steps 7-12"], transformer_params, transformer_time),
    ("Hybrid split 1-12", comparison["Hybrid split steps 1-12"], hybrid_params, hybrid_time),
    ("MLP full 1-12", comparison["MLP full steps 1-12"], mlp_full_params, mlp_full_time),
    ("Transformer full 1-12", comparison["Transformer full steps 1-12"], transformer_full_params, transformer_full_time),
]

print(f"{'Model':<14} {'MAE':>10} {'MSE':>10} {'Params':>12} {'Avg batch sec':>15}")
print("-" * 65)
for model_name, metrics, params, avg_time in rows:
    print(f"{model_name:<14} {metrics['MAE']:>10.6f} {metrics['MSE']:>10.6f} {params:>12,} {avg_time:>15.6f}")
